In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Phase6").getOrCreate()

# =========================
# Dirty Customers Dataset
# =========================

customers_data = [
    (1, "John Doe", "john@example.com", "Hyderabad"),
    (2, "Alice ", "alice@example.com", "Chennai"),
    (3, None, "bob@example.com", "Bangalore"),        # NULL name
    (4, "David", None, "Mumbai"),                    # NULL email
    (5, "Eva", "eva@example.com", "Hyderabad"),
    (6, "Frank", "frank@example.com", "Delhi"),
]

customers1 = spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city"])

# =========================
# Dirty Orders Dataset
# =========================

orders_data = [
    (101, 1, "2024-01-01", 1000),
    (102, 2, "2024-01-02", 2000),
    (103, 3, "2024-01-03", -500),     # INVALID negative value
    (104, 99, "2024-01-04", 1500),    # INVALID FK (customer_id 99)
    (105, 1, "2024-01-05", None),     # NULL amount
    (106, 5, "2024-01-06", 3000),
    (107, 5, "2024-01-07", 3000),     # duplicate-like record
]

orders1 = spark.createDataFrame(orders_data, ["order_id", "customer_id", "order_date", "amount"]);

In [0]:
orders1 = orders1.withColumn("order_date", to_date(col("order_date")))

print("Initial Data:")
customers1.show()
orders1.show()

Initial Data:
+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|  Alice |alice@example.com|  Chennai|
|          3|    NULL|  bob@example.com|Bangalore|
|          4|   David|             NULL|   Mumbai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2024-01-01|  1000|
|     102|          2|2024-01-02|  2000|
|     103|          3|2024-01-03|  -500|
|     104|         99|2024-01-04|  1500|
|     105|          1|2024-01-05|  NULL|
|     106|          5|2024-01-06|  3000|
|     107|          5|2024-01-07|  3000|
+--------+-----------+----------+------+



#### STEP 1: CLEAN DATA

### remove nulls


In [0]:
customers_clean = customers1.filter(col("name").isNotNull()) \
                           .filter(col("email").isNotNull())

### trim names

In [0]:
customers_clean = customers_clean.withColumn("name", trim(col("name")))

### remove invalid amounts

In [0]:
orders_clean = orders1.filter(col("amount").isNotNull()) \
                     .filter(col("amount") > 0)

print("After Cleaning:")
customers_clean.show()
orders_clean.show()

After Cleaning:
+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|   Alice|alice@example.com|  Chennai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2024-01-01|  1000|
|     102|          2|2024-01-02|  2000|
|     104|         99|2024-01-04|  1500|
|     106|          5|2024-01-06|  3000|
|     107|          5|2024-01-07|  3000|
+--------+-----------+----------+------+



#### STEP 2: VALIDATE DATA

### invalid foreign keys

In [0]:
invalid_fk = orders_clean.join(customers_clean, "customer_id", "left_anti")

print("Invalid Customer IDs:")
invalid_fk.show()

Invalid Customer IDs:
+-----------+--------+----------+------+
|customer_id|order_id|order_date|amount|
+-----------+--------+----------+------+
|         99|     104|2024-01-04|  1500|
+-----------+--------+----------+------+



#### STEP 3: JOIN DATASETS

In [0]:
inner_join = orders_clean.join(customers_clean, "customer_id", "inner")
left_join = orders_clean.join(customers_clean, "customer_id", "left")

print("Inner Join:")
inner_join.show()

print("Left Join:")
left_join.show()

print("Row Count Comparison:")
print("Inner:", inner_join.count())
print("Left:", left_join.count())

Inner Join:
+-----------+--------+----------+------+--------+-----------------+---------+
|customer_id|order_id|order_date|amount|    name|            email|     city|
+-----------+--------+----------+------+--------+-----------------+---------+
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|
|          2|     102|2024-01-02|  2000|   Alice|alice@example.com|  Chennai|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|
+-----------+--------+----------+------+--------+-----------------+---------+

Left Join:
+-----------+--------+----------+------+--------+-----------------+---------+
|customer_id|order_id|order_date|amount|    name|            email|     city|
+-----------+--------+----------+------+--------+-----------------+---------+
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|
|          2|     102|2024-01-02|  2000|

#### STEP 4: TRANSFORMATIONS

In [0]:
agg_df = inner_join.groupBy("customer_id", "name", "city") \
    .agg(
        sum("amount").alias("total_spend"),
        count("order_id").alias("order_count")
    )

print("Aggregated Data:")
agg_df.show()

Aggregated Data:
+-----------+--------+---------+-----------+-----------+
|customer_id|    name|     city|total_spend|order_count|
+-----------+--------+---------+-----------+-----------+
|          1|John Doe|Hyderabad|       1000|          1|
|          2|   Alice|  Chennai|       2000|          1|
|          5|     Eva|Hyderabad|       6000|          2|
+-----------+--------+---------+-----------+-----------+



#### STEP 5: WINDOW FUNCTIONS

### rank customers by spend

In [0]:
window = Window.orderBy(col("total_spend").desc())
ranked = agg_df.withColumn("rank", rank().over(window))

print("Ranked Customers:")
ranked.show()

Ranked Customers:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------+---------+-----------+-----------+----+
|customer_id|    name|     city|total_spend|order_count|rank|
+-----------+--------+---------+-----------+-----------+----+
|          5|     Eva|Hyderabad|       6000|          2|   1|
|          2|   Alice|  Chennai|       2000|          1|   2|
|          1|John Doe|Hyderabad|       1000|          1|   3|
+-----------+--------+---------+-----------+-----------+----+



### running total

In [0]:
window_running = Window.orderBy("customer_id")

running = agg_df.withColumn("running_total", sum("total_spend").over(window_running))

print("Running Total:")
running.show()

Running Total:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------+---------+-----------+-----------+-------------+
|customer_id|    name|     city|total_spend|order_count|running_total|
+-----------+--------+---------+-----------+-----------+-------------+
|          1|John Doe|Hyderabad|       1000|          1|         1000|
|          2|   Alice|  Chennai|       2000|          1|         3000|
|          5|     Eva|Hyderabad|       6000|          2|         9000|
+-----------+--------+---------+-----------+-----------+-------------+



In [0]:
window_lag = Window.partitionBy("customer_id").orderBy("customer_id")

lag_df = inner_join.withColumn("prev_amount", lag("amount").over(window_lag))

print("Lag Function:")
lag_df.show()

Lag Function:
+-----------+--------+----------+------+--------+-----------------+---------+-----------+
|customer_id|order_id|order_date|amount|    name|            email|     city|prev_amount|
+-----------+--------+----------+------+--------+-----------------+---------+-----------+
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|       NULL|
|          2|     102|2024-01-02|  2000|   Alice|alice@example.com|  Chennai|       NULL|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|       NULL|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|       3000|
+-----------+--------+----------+------+--------+-----------------+---------+-----------+



#### top 3 customers per city

In [0]:
# total spend per customer
window = Window.partitionBy("city").orderBy(col("total_spend").desc())
top3 = agg_df.withColumn("rank", row_number().over(window)) \
    .filter(col("rank") <= 3)

top3.show()

+-----------+--------+---------+-----------+-----------+----+
|customer_id|    name|     city|total_spend|order_count|rank|
+-----------+--------+---------+-----------+-----------+----+
|          2|   Alice|  Chennai|       2000|          1|   1|
|          5|     Eva|Hyderabad|       6000|          2|   1|
|          1|John Doe|Hyderabad|       1000|          1|   2|
+-----------+--------+---------+-----------+-----------+----+



#### STEP 6: DATE ANALYSIS

### extract month from date

In [0]:
df_month = inner_join.withColumn("month", month("order_date"))

df_month.show()

+-----------+--------+----------+------+--------+-----------------+---------+-----+
|customer_id|order_id|order_date|amount|    name|            email|     city|month|
+-----------+--------+----------+------+--------+-----------------+---------+-----+
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|    1|
|          2|     102|2024-01-02|  2000|   Alice|alice@example.com|  Chennai|    1|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|    1|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|    1|
+-----------+--------+----------+------+--------+-----------------+---------+-----+



#### monthly sales aggregation

In [0]:
monthly_sales = df_month.groupBy("month") \
    .agg(sum("amount").alias("total_sales")) \
    .orderBy("month")

print("Monthly Sales:")
monthly_sales.show()

Monthly Sales:
+-----+-----------+
|month|total_sales|
+-----+-----------+
|    1|       9000|
+-----+-----------+



### calculate difference between dates

In [0]:
window = Window.partitionBy("customer_id").orderBy("order_date")

df_diff = inner_join.withColumn(
    "previous_date",
    lag("order_date").over(window)
).withColumn(
    "date_difference",
    datediff(col("order_date"), col("previous_date"))
)

print("Date Difference:")
df_diff.show()

Date Difference:
+-----------+--------+----------+------+--------+-----------------+---------+-------------+---------------+
|customer_id|order_id|order_date|amount|    name|            email|     city|previous_date|date_difference|
+-----------+--------+----------+------+--------+-----------------+---------+-------------+---------------+
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|         NULL|           NULL|
|          2|     102|2024-01-02|  2000|   Alice|alice@example.com|  Chennai|         NULL|           NULL|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|         NULL|           NULL|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|   2024-01-06|              1|
+-----------+--------+----------+------+--------+-----------------+---------+-------------+---------------+



### trend analysis by month

In [0]:
trend_analysis = df_month.groupBy("month") \
    .agg(
        sum("amount").alias("total_sales"),
        avg("amount").alias("avg_sales")
    ) \
    .orderBy("month")

print("Trend Analysis:")
trend_analysis.show()

Trend Analysis:
+-----+-----------+---------+
|month|total_sales|avg_sales|
+-----+-----------+---------+
|    1|       9000|   2250.0|
+-----+-----------+---------+



#### STEP 7:FINAL PIPELINE

In [0]:
final_df = ranked.join(monthly_sales)

print("Final Output:")
final_df.show()

Final Output:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------+---------+-----------+-----------+----+-----+-----------+
|customer_id|    name|     city|total_spend|order_count|rank|month|total_sales|
+-----------+--------+---------+-----------+-----------+----+-----+-----------+
|          5|     Eva|Hyderabad|       6000|          2|   1|    1|       9000|
|          2|   Alice|  Chennai|       2000|          1|   2|    1|       9000|
|          1|John Doe|Hyderabad|       1000|          1|   3|    1|       9000|
+-----------+--------+---------+-----------+-----------+----+-----+-----------+



#### STEP 8: SAVE OUTPUT

In [0]:
final_df.write.mode("overwrite").csv("/Volumes/workspace/phase5_schema/volume_phase5/phase6_output")
print("Pipeline Completed Successfully")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Pipeline Completed Successfully
